###Shrinkage LDA tutorial

This Jupyter notebook contains a introduction to Shrinkage LDA (sLDA). It assumes that the reader is already familiar with regular LDA. 

**Author:** Tom 
**Date:** 2026-04-23



sLDA is an extension of regular LDA that allows classification on data with relatively many features, compared to the amount samples. It can be seen as an extra regulaization step on top of regular LDA. sLDA includes a bias that tries to transform the original covariance matrix such that it becomes more similar to the identity matrix. This helps generalize any covariance matrix such that overfitting can be countered. It does come at a cost in the form of the bias term. This term specifies how strongly the covariance matrix is transformed in the direction of the identity matrix. 

Below, you can find a full sLDA pipeline that is applied to dummy data.

First install the necessary python libraries

In [2]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

Create dummy data. Remember that LDA relies on 2 classes that have the same covariance matrix with different means.

In [17]:
n_samples = 20
n_features = 1000  # high-dimensional
mean_diff = 0.05 # This parameter defines how separable the classes are

A = np.random.randn(n_features, n_features)
Sigma = A @ A.T
Sigma /= np.max(np.diag(Sigma))

mu_0 = np.zeros(n_features)
mu_1 = np.ones(n_features) * mean_diff

X0 = np.random.multivariate_normal(mu_0, Sigma, n_samples) # Class 1
X1 = np.random.multivariate_normal(mu_1, Sigma, n_samples) # Class 2

X = np.vstack([X0, X1])
y = np.array([0]*n_samples + [1]*n_samples)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


Next we will train both regular LDA and sLDA to see how both versions compare to the high dimensional data.

In [18]:
# --------------------
# 1. Standard LDA
# --------------------
lda = LinearDiscriminantAnalysis(solver="svd")
lda.fit(X_train, y_train)
pred_lda = lda.predict(X_test)

# --------------------
# 2. Shrinkage LDA (sLDA)
# --------------------
slda = LinearDiscriminantAnalysis(
    solver="lsqr",
    shrinkage="auto"
)
slda.fit(X_train, y_train)
pred_slda = slda.predict(X_test)

# --------------------
# Results
# --------------------
print("LDA accuracy:", accuracy_score(y_test, pred_lda))
print("sLDA accuracy:", accuracy_score(y_test, pred_slda))

LDA accuracy: 0.4166666666666667
sLDA accuracy: 0.75
